In [1]:
import sys, pathlib
_here = pathlib.Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "src" / "stats_textbook").exists():
        sys.path.insert(0, str(_p / "src"))
        break

# 統計的推測の風景 — 不確実性を測り、判断する言語

> 各章は 直感 → 図 → 最小限の数式 → Python 実装 → 実験 → 演習 の順。
> 本文は日本語、コードは英語、数式に日本語を入れない。

## なぜ同じデータから 2 人が違う結論を出すのか

硬貨を 10 回投げて 8 回表が出た。この硬貨は偏っているだろうか。

一方は言う。「8 回も表が出たのだから偏っている」。
もう一方は言う。「10 回では何も言えない」。

奇妙なのは、**この 2 人が見ているデータはまったく同じ**だということである。
計算間違いをしているわけでも、情報を隠しているわけでもない。
それでも結論が割れる。

割れる理由は、データそのものの中にはない。
「8 回が珍しいかどうか」を判断するには、
**何と比べて珍しいのか** という基準が要る。
その基準はデータには書かれておらず、こちらが持ち込むしかない。

公正な硬貨だったら 8 回以上表が出ることはどれくらい起きるのか。
それを知らなければ「珍しい」とは言えない。数えてみよう。

In [2]:
import numpy as np
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"

from stats_textbook import datasets, distributions, plotting, processes, simulation

RANDOM_SEED = 0
print("setup ok")

setup ok


In [3]:
# 上の話に出てきたデータ。seed 25 の公正な硬貨がたまたま 8 回表を出した。
flips = datasets.coin_flips(10, p=0.5, seed=25)
print("観測:", flips, "-> 表", flips.sum(), "回")
assert flips.sum() == 8, "本文が語っているのは 8 回表のデータである"

# 公正な硬貨でも 8 回以上表が出ることは珍しくない。
reps = simulation.sampling_distribution(
    np.sum, lambda n, rng: (rng.random(n) < 0.5).astype(int), n=10, n_reps=20_000, seed=0
)
print(f"公正な硬貨で 8 回以上表が出る割合: {(reps >= 8).mean():.3f}")

観測: [1 1 1 1 1 1 0 0 1 1] -> 表 8 回
公正な硬貨で 8 回以上表が出る割合: 0.055


20 回に 1 回ほどは起きる。「まず起こらないこと」ではない。

しかも上のデータは、**実際に公正な硬貨から出ている**。
`p=0.5` を指定して生成したのだから、偏りは無いと私たちは知っている。
それでも 8 回表が出た。

つまり「8 回表」というデータは、公正な硬貨という仮説とそれほど矛盾していない。
かといって「偏っている」という仮説とも矛盾しない。
**データが両方の仮説と両立してしまう** ため、どちらを選ぶかはデータの外側で決まる。

本書は、この「外側」を明示的に扱う道具を作る。
ただし注意してほしい。私たちがここで「公正だと知っている」のは、
データを生成した側に回ったからである。
現実の推測では、真値を知る側には立てない。
だからこそ本書は繰り返し、**真値が分かっている人工データで手続きを検算する**。
手続きが正しく動くかどうかは、真値を知っている場合にしか確かめられない。

```{admonition} 核心 — ひとことで
:class: tip
データが珍しいかどうかは、データだけでは決まらない。
何と比べて珍しいのか、という基準をこちらが持ち込んで初めて決まる。
本書の第Ⅰ部はその基準を作る道具、第Ⅱ部はそれを使って判断する方法である。
```

## 本書の地図

### 第Ⅰ部 確率論 — 基準を作る

| 章 | 何を与えるか | 何に依存するか |
|---|---|---|
| 01 確率の土台 | 条件付き確率とベイズの定理を計算道具として | — |
| 02 確率変数と期待値 | 分布を数値に潰す操作。期待値・分散・条件付き期待値 | 01 |
| 03 分布の動物園 | 主要分布の関係と指数型分布族・十分統計量 | 02 |
| 04 極限定理 | 大数の法則・中心極限定理・収束の 3 種・デルタ法 | 02, 03 |
| 05 確率過程 | 時間軸が入るとどうなるか。マルコフ性・定常分布 | 02 |

### 第Ⅱ部 統計的推測 — 基準を使って判断する

| 章 | 何を与えるか | 何に依存するか |
|---|---|---|
| 06 推定と最尤法 | 推定量の良さの定義・MLE・Fisher 情報・漸近正規性 | 03, 04 |
| 07 信頼区間とブートストラップ | 区間推定の正しい読み方・被覆確率の実測 | 04, 06 |
| 08 仮説検定 | 検定の構造・p 値・検出力・多重比較 | 04, 06 |
| 09 回帰の推測 | 回帰係数の分布・残差診断・頑健標準誤差 | 06, 08 |
| 10 一般化線形モデル | 指数型分布族から GLM へ・IRLS | 03, 09 |
| 11 頻度論とベイズ | 同じデータを両流儀で解いて比べる | 06, 07, 08 |

第Ⅰ部で作る地図を、先に見ておこう。
主要な分布は独立した暗記項目ではなく、少数の関係でつながっている。

In [4]:
plotting.relation_graph()

## 本書を貫く 3 原則

1. **確率は長期頻度で定義する** — だから信頼区間は「真値が入る確率」ではない(07 章)
2. **すべての主張はシミュレーションで検算する** — 被覆確率も第 1 種の誤り率も実測する(07・08 章)
3. **モデルは仮定の束であり、診断せずに使わない**(09・10 章)

2 番目が本書の性格を決めている。
「この区間は 95% の確率で真値を含む」といった主張は、
長期頻度についての予言である。
予言なら実測できる。本書では**実測する**。

```{admonition} 実社会では
:class: note
医薬品の承認、A/B テストの採否、工場の出荷判定。
いずれも「観測された差が偶然の範囲か」を決める手続きに支えられている。
その手続きが何を保証し、何を保証しないのかを読めるようになるのが本書の目的である。
```

## 読み方

### 章の構成

各章は次の順で進む。

1. 導入 — 具体的な問いや誤解から入る(数式なし)
2. 直感と図 — インタラクティブな図で現象を見る
3. 定式化 — 最小限の数式。定義と主張を分ける
4. 実装 — `stats_textbook` のコードを呼ぶ
5. 実験 — シミュレーションで主張を検算する
6. 落とし穴 — 典型的な誤用と、それが数値でどう現れるか
7. 演習 — 3 から 5 問(解答は 13 章)

### 記号の約束

- $X, Y$ は確率変数、$x, y$ はその実現値
- $\theta$ は母数(推定したい未知の量)、$\hat\theta$ はその推定量
- $n$ は標本サイズ、$N$ はシミュレーションの反復回数
- $P(\cdot)$ は確率、$E[\cdot]$ は期待値、$\mathrm{Var}(\cdot)$ は分散

### 実行環境

すべてのデータは合成で、外部ダウンロードは一切ない。
乱数は seed 固定なので、手元で実行しても本書と同じ数値が出る。
図は静的な HTML でもスライダーが動く。

### 姉妹本

- ベイズ流の推論は `analytics/bayesian`(ベイズ推定の体験)
- 予測性能の検証は `analytics/machine_learning`(機械学習の実践)
- 本書 11 章はその両方への橋渡しになっている

いずれも同じ analytics シリーズの一冊で、`analytics/README.md` から辿れる。